# 06 앱 전체 실행 (코랩, 로컬 PC 없이)
드라이브의 파일로 **앱을 처음부터 끝까지** 돌려 본다. 런타임은 **CPU로 충분** (서버도 CPU에서 돈다).

1. 저장소 받기 + 드라이브 연결
2. 앱 파일 복사: 분류 모델 `best/convnext_base_exp004/best.pt` (한솔 최종 모델), 검색 목록 `best/clip_base/` 두 파일
3. (선택) LLM 키: 왼쪽 🔑 **보안 비밀**에 `GEMINI_API_KEY` (또는 `LLM_PROVIDER=groq` + `GROQ_API_KEY`) 추가하고 노트북 액세스 켜기. 없으면 설명·AI 추정 이름만 빠짐
4. 자체 검사 (`knowledge.judge`, `knowledge.search`)
5. 서버 켜기 → 레시피 사진으로 `/analyze` 끝까지 확인 (아는 음식 ramen, 처음 보는 음식 khao_soi)
6. 화면 열기 (본인 구글 계정에서만 열리는 링크)

팀원 계정이면 먼저 드라이브 "공유 문서함"의 `26-2 딥러닝`을 **내 드라이브에 바로가기 추가**.

## 0. 드라이브 연결 · 코드 받기

In [ ]:
import os, sys, io, glob, math, time, random, unicodedata, subprocess
from google.colab import drive
drive.mount('/content/drive')

def find(parent, name):
    for d in os.listdir(parent):
        if unicodedata.normalize('NFC', d).strip() == name:
            return f'{parent}/{d}'
    raise FileNotFoundError(f"'{parent}' 안에 '{name}' 없음. 보이는 폴더: {sorted(os.listdir(parent))}")

TEAM = find('/content/drive/MyDrive', '26-2 딥러닝')
DATA = f'{TEAM}/data1'
IMG = f'{DATA}/images'                       # 03에서 모은 사진
RECIPES = f'{DATA}/table_recipes.parquet'    # 03에서 만든 레시피 (제목·재료)
EXP = find(TEAM, '찍먹_실험기록')

REPO = '/content/deeplearning_5'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yura103/deeplearning_5', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
sys.path.insert(0, REPO)

import torch
print('준비 완료 (GPU 없어도 됨: 서버는 CPU로 돌림)', torch.cuda.is_available())

## 1. 앱 파일 복사 + 설치

In [ ]:
import shutil
W, IDX = f'{REPO}/modeling/weights', f'{REPO}/knowledge/index'
os.makedirs(W, exist_ok=True); os.makedirs(IDX, exist_ok=True)
MODEL = 'convnext_base_exp004'   # 한솔 최종 모델 exp004 (test 93.46%, 12에폭 최고). 예전: 'convnext_base'(exp003) / 'convnext_tiny'
shutil.copy(f'{EXP}/best/{MODEL}/best.pt', f'{W}/best.pt')
for f in ('index_emb.npy', 'index_meta.parquet'):
    shutil.copy(f'{EXP}/best/clip_base/{f}', f'{IDX}/{f}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/requirements.txt'], check=True)
print('복사·설치 완료:', sorted(os.listdir(W)), sorted(os.listdir(IDX)))

## 2. (선택) LLM 키 — 보안 비밀에서 읽기

In [ ]:
from google.colab import userdata
for k in ('GEMINI_API_KEY', 'GROQ_API_KEY', 'LLM_PROVIDER', 'GEMINI_MODEL', 'GROQ_MODEL'):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v          # 값은 출력하지 않음
    except Exception:                  # 없음 / 노트북 액세스 꺼짐
        pass
print('LLM 제공자:', os.environ.get('LLM_PROVIDER', 'gemini'),
      '/ 키 있음:', any(k in os.environ for k in ('GEMINI_API_KEY', 'GROQ_API_KEY')))

## 3. 자체 검사

In [ ]:
for mod in ('knowledge.judge', 'knowledge.search'):
    r = subprocess.run([sys.executable, '-m', mod], cwd=REPO, capture_output=True, text=True)
    print(f'[{mod}] 종료 코드 {r.returncode}')
    print((r.stdout + r.stderr)[-1500:])
    assert r.returncode == 0, f'{mod} 자체 검사 실패'

## 4. 서버 켜기

In [ ]:
import requests
LOG = '/content/server.log'
subprocess.run(['pkill', '-f', 'uvicorn backend.main:app'])   # 다시 돌릴 때 이전 서버 정리
srv = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.main:app', '--port', '8000'],
                       cwd=REPO, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, env=os.environ.copy())
for _ in range(60):
    try:
        print('알레르기 목록:', requests.get('http://localhost:8000/allergens', timeout=2).json()); break
    except Exception:
        time.sleep(1)
else:
    print(open(LOG).read()); raise RuntimeError('서버가 안 켜짐 (위 로그 확인)')

## 5. 사진으로 끝까지 확인

In [ ]:
import json
import pandas as pd
from PIL import Image
from IPython.display import display

def photo(label, i=0):
    """드라이브 레시피 사진 (03에서 모은 것) 한 장 → jpg bytes"""
    return pd.read_parquet(f'{IMG}/{label}.parquet', columns=['jpg']).jpg.iloc[i]

results = {}
for label in ['ramen', 'khao_soi']:   # 아는 음식(Food-101) / 처음 보는 음식(74종 중 하나)
    jpg = photo(label)
    display(Image.open(io.BytesIO(jpg)).resize((160, 160)))
    t = time.time()
    r = requests.post('http://localhost:8000/analyze', data={'allergens': '밀,땅콩,새우'},
                      files={'image': ('photo.jpg', jpg, 'image/jpeg')}, timeout=600)
    print(f'{label}: HTTP {r.status_code}, {time.time() - t:.1f}초')
    results[label] = r.json() if r.ok else r.text
    print(json.dumps(results[label], ensure_ascii=False, indent=1)[:2500], '\n')

## 6. 2단계(CLIP 검색 + LLM 교차 확인) 직접 확인

In [ ]:
# 2단계를 직접 확인 (사진에 따라 위에서 1단계로 갈 수도 있어서, 검색 + 교차 확인을 따로 한 번 더 부름)
from knowledge import search
from knowledge.judge import judge
from llm.guess import guess_name, guess_ingredients
img = Image.open(io.BytesIO(photo('khao_soi')))
t = time.time(); hit = search.search(img); print(f'CLIP 검색 {time.time() - t:.1f}초, 비슷한 음식:', hit[1][:5])
g = guess_name(img); print('LLM 이름 추정:', g)
r2, foods = search.crosscheck(hit, g, guess_ingredients)
print('근거:', r2['source']); print('판정:', judge(r2, ['밀', '땅콩', '새우'])['verdict'])

## 7. 화면 열기

In [ ]:
# 화면 열기: 노트북 안에 앱 화면이 뜬다 (본인 구글 계정에서만). 새 창 방식(serve_kernel_port_as_window)은 404가 나기 쉬움
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=900)

## 서버 로그

In [ ]:
# 문제가 있으면 서버 로그 확인
print(open(LOG).read()[-4000:])

## 끝
- 모두 통과하면 `results`의 `route`(1단계/2단계), `verdict`, `explanation`이 기대대로인지 본다.
- 오류가 나면 마지막 셀의 서버 로그를 그대로 공유.
- 서버 끄기: `srv.terminate()`